# 12 — Polarizzazione dei lettori NYT nel tempo

Misuriamo come è cambiata nel tempo la **polarizzazione dei commenti dei lettori** sugli articoli ICE, attraverso i quattro periodi politici usati. Le etichette di posizione ed emozione vengono dal notebook 11 §2 (Claude Haiku 5.5, prompt v3).

Due livelli:
1. **Per periodo** (§1-§2): come si distribuiscono le posizioni (critica, neutra, favorevole) e quanto sono lontane e bilanciate.
2. **Per topic dell'articolo** (§3): quali argomenti producono commenti più critici, più polarizzati o più carichi di emozioni negative rispetto agli altri articoli dello stesso periodo.

## 0. Setup

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
PLOTS_DIR = PROCESSED_DIR / "plots"
IN_COMMENT_SENTIMENT = PROCESSED_DIR / "comments_ice_sentiment.parquet"
OUT_POLARIZATION = PROCESSED_DIR / "comments_polarization_by_period.csv"

SEED = 42
N_BOOT = 1000
PERIODS = ["2003-2016", "2017-2020", "2021-2024", "2025-presente"]
STANCES = ["crit", "neu", "fav"]
STANCE_SCORE = {"crit": -1, "neu": 0, "fav": 1}
STANCE_FULL = {"crit": "critical", "neu": "neutral", "fav": "favorable"}
EMOTIONS = ["anger", "contempt", "fear", "sadness", "hope", "none"]
NEGATIVE_EMOTIONS = {"anger", "contempt", "fear", "sadness"}

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("nyt_polarization")

# stessi colori dei notebook 09 e 11
PERIOD_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
STANCE_COLORS = {"crit": "#d64545", "neu": "#b8b8b0", "fav": "#2a78d6"}
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e6e1",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
})
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Preparazione dei dati

- Si tengono solo i commenti **pertinenti**: si tolgono i not_about.
- I periodi restano in ordine cronologico (categoria ordinata).
- La posizione diventa un numero: critica = −1, neutra = 0, favorevole = +1. Serve per media e varianza (§2).

In [ ]:
comments = pd.read_parquet(IN_COMMENT_SENTIMENT, columns=["comment_id", "article_id", "user_key", "type", "period",
                                                          "llm_stance", "llm_emotion"])
pertinent = comments[ comments["llm_stance"].isin(STANCES) & comments["llm_emotion"].notna() ].copy()
pertinent["period"] = pd.Categorical(pertinent["period"].astype(str), categories=PERIODS, ordered=True)
pertinent["stance"] = pd.Categorical(pertinent["llm_stance"], categories=STANCES, ordered=True)
pertinent["score"] = pertinent["llm_stance"].map(STANCE_SCORE)
pertinent["negative"] = pertinent["llm_emotion"].isin(NEGATIVE_EMOTIONS)
log.info("Commenti etichettati: %d; pertinenti: %d (%.0f%%); not_about tolti: %d", len(comments), len(pertinent),
         len(pertinent) / len(comments) * 100, (comments["llm_stance"] == "not_about").sum())

(pertinent.groupby("period", observed=True)
 .agg(commenti=("comment_id", "size"), articoli=("article_id", "nunique"), utenti=("user_key", "nunique")))

## 2. Polarizzazione ideologica

### 2.1 Distribuzione delle posizioni per periodo

Quota di commenti critici, neutri e favorevoli in ogni periodo (sui commenti pertinenti). Polarizzazione strutturale significa che il centro (neutri) si svuota e i due estremi hanno pesi simili, cioè una distribuzione a "U". Se invece cresce uno solo dei due estremi, l'opinione converge da una parte: meno polarizzazione, non di più.

In [ ]:
stance_share = (pd.crosstab(pertinent["period"], pertinent["stance"], normalize="index")
                .reindex(index=PERIODS, columns=STANCES))

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.26
x = np.arange(len(PERIODS))
for k, s in enumerate(STANCES):
    bars = ax.bar(x + (k - 1) * width, stance_share[s], width, color=STANCE_COLORS[s], label=STANCE_FULL[s])
    ax.bar_label(bars, labels=[f"{v:.0%}" for v in stance_share[s]], fontsize=8, padding=2)
ax.set_xticks(x, PERIODS)
ax.set_ylim(0, 0.85)
ax.set_ylabel("Share of pertinent comments")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.legend(frameon=False, fontsize=9)
ax.set_title("Reader stance toward ICE and immigration enforcement, by period", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "polarization_stance_by_period.png", dpi=150, bbox_inches="tight")
plt.show()

Per ogni periodo tre barre (critici, neutri, favorevoli) che sommano a 100%. Una "U" sono due barre esterne alte e simili con la barra centrale bassa; i neutri sono comunque bassi in tutti i periodi anche per costruzione (il prompt assegna `neu` solo ai commenti che non pendono da nessuna parte).

### 2.2 Indici di polarizzazione nel tempo

Tre misure per periodo, tutte tra 0 e 1:
- **Varianza della posizione** (indice principale): varianza del punteggio −1/0/+1. Vale 1 quando metà dei commenti è critica e metà favorevole (massima polarizzazione), 0 quando tutti hanno la stessa posizione (la misura di polarizzazione per dispersione di DiMaggio, Evans e Bryson (1996), e cattura insieme le due condizioni: estremi pieni e campi bilanciati).
- **Equilibrio tra i campi**: 1 − |crit − fav| / (crit + fav). Vale 1 quando critici e favorevoli sono lo stesso numero, 0 quando c'è un solo campo.
- **Quota degli estremi** (crit + fav, cioè 1 − neutri): l'indice proposto inizialmente, tenuto come controllo. Dipende molto dalla regola del prompt sui neutri, quindi da solo non distingue un dibattito spaccato da uno unanime.

**Intervalli di confidenza al 95%**: bootstrap per articolo (N_BOOT = 1.000). In ogni replica si ricampionano con reinserimento gli articoli del periodo, con tutti i loro commenti, e si ricalcolano le misure: così l'intervallo tiene conto che i commenti dello stesso articolo si somigliano e che pochi articoli possono pesare molto.

In [ ]:
def polarization_metrics(counts: np.ndarray) -> dict:
    """counts = [crit, neu, fav] sommati sui commenti."""
    crit, neu, fav = counts
    n = counts.sum()
    mean = (fav - crit) / n
    return {"variance": (fav + crit) / n - mean ** 2,
            "balance": 1 - abs(crit - fav) / (crit + fav),
            "extremes": (crit + fav) / n,
            "mean_score": mean}


rng = np.random.default_rng(SEED)
rows = []
for period in PERIODS:
    per_article = (pertinent[pertinent["period"] == period]
                   .groupby(["article_id", "stance"], observed=False).size().unstack(fill_value=0)[STANCES].to_numpy())
    point = polarization_metrics(per_article.sum(axis=0))
    boot = pd.DataFrame([polarization_metrics(per_article[rng.integers(0, len(per_article), len(per_article))].sum(axis=0))
                         for _ in range(N_BOOT)])
    for metric, value in point.items():
        lo, hi = boot[metric].quantile([0.025, 0.975])
        rows.append({"period": period, "metric": metric, "value": value, "ci_low": lo, "ci_high": hi,
                     "comments": int(per_article.sum()), "articles": len(per_article)})
polarization = pd.DataFrame(rows)
polarization.to_csv(OUT_POLARIZATION, index=False)
log.info("Scritto %s", OUT_POLARIZATION.relative_to(BASE_DIR))

metric_style = {"variance": ("Stance variance (main index)", "#1f1f1f", "-"),
                "balance": ("Balance between camps", "#8e44ad", "-"),
                "extremes": ("Share of extremes (crit + fav)", "#9a9a92", "--")}
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(PERIODS))
for metric, (label, color, ls) in metric_style.items():
    m = polarization[polarization["metric"] == metric].set_index("period").loc[PERIODS]
    ax.plot(x, m["value"], marker="o", color=color, linestyle=ls, label=label)
    ax.fill_between(x, m["ci_low"], m["ci_high"], color=color, alpha=0.12, linewidth=0)
ax.set_xticks(x, PERIODS)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Index (0-1)")
ax.legend(frameon=False, fontsize=8, loc="lower left")
ax.set_title("Polarization of reader stance by period (95% CI, article bootstrap)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "polarization_index_by_period.png", dpi=150, bbox_inches="tight")
plt.show()

polarization.pivot(index="period", columns="metric", values="value").loc[PERIODS].round(3)

- Linea nera = stance variance, indica la varianza del punteggio -1/0/1. Vale 1 quando abbiamo la massima spaccatura tra i commenti (es. metà dei commenti critica e metà favorevole) e 0 quando tutti hanno la stessa posizione. Dunque più il valore è basso più l'opinione è uniforme.
- Linea viole = balance between camps: misura 1 - |crit - fav| / (crit + fav), cioè guarda il rapporto tra critici e favorevoli (senza contare i neutri). Vale se i due campi sono uguali e 0 se esiste un solo campo (è un indicatore secondario di supporto).
- Linea grigia: share of extremes, misura crit + fav (1 - neutri). Ci dice poco perchè il prompt assegna neutro solo se il commento non pende da nessuna parte, quindi i neutri sono pochi per costruzione. Serve da controllo

## 3. Polarizzazione per topic dell'articolo

Ogni commento eredita il topic del suo articolo: i 25 topic BERTopic (mpnet) del notebook 04, gli stessi del notebook 09. Esclusi i commenti su articoli outlier (topic −1).

**Selezione**: solo i topic con almeno MIN_TOPIC_ARTICLES = 10 articoli con commenti pertinenti, perché con pochi articoli il risultato dipende da uno o due thread.

**Confronto al netto del periodo.** Topic e periodo si sovrappongono: molti topic esistono quasi solo nel 2025+ (sparatorie, proteste, aeroporti), e il 2025+ è già il periodo più critico e meno polarizzato. Per ogni topic si calcola quindi un valore **atteso**: quello che il topic avrebbe se i suoi commenti avessero la distribuzione media del loro periodo (media dei periodi pesata con la quota di commenti del topic in ciascun periodo; standardizzazione indiretta). La differenza osservato − atteso dice se il topic genera commenti diversi **rispetto agli altri articoli dello stesso periodo**. Tre misure:
- **posizione media** (−1 tutti critici, +1 tutti favorevoli): differenza > 0 = più favorevole del previsto;
- **varianza della posizione** (§2.2): differenza > 0 = più polarizzato del previsto;
- **quota di emozioni negative**: differenza > 0 = più carico emotivamente del previsto.

In [ ]:
IN_ARTICLE_TOPICS = PROCESSED_DIR / "articles_ice_topics.parquet"
OUT_TOPIC_POLARIZATION = PROCESSED_DIR / "comments_polarization_by_topic.csv"
MIN_TOPIC_ARTICLES = 10

article_topics = pd.read_parquet(IN_ARTICLE_TOPICS, columns=["id", "topic", "topic_name"]).rename(columns={"id": "article_id"})
by_topic = pertinent.merge(article_topics, on="article_id", how="left")
log.info("Commenti pertinenti su articoli outlier (esclusi): %d", (by_topic["topic"] == -1).sum())
by_topic = by_topic[by_topic["topic"] != -1]

topic_articles = by_topic.groupby("topic_name")["article_id"].nunique()
kept_topics = topic_articles[topic_articles >= MIN_TOPIC_ARTICLES].index
by_topic = by_topic[by_topic["topic_name"].isin(kept_topics)]
log.info("Topic tenuti: %d su %d (≥ %d articoli); commenti: %d", len(kept_topics), len(topic_articles),
         MIN_TOPIC_ARTICLES, len(by_topic))

# Distribuzione media di ogni periodo, su tutti i commenti pertinenti (riferimento per l'atteso)
period_shares = pd.crosstab(pertinent["period"], pertinent["stance"], normalize="index").reindex(index=PERIODS, columns=STANCES).to_numpy()
period_negative = pertinent.groupby("period", observed=True)["negative"].mean().reindex(PERIODS).to_numpy()


def shares_metrics(shares: np.ndarray) -> tuple[float, float]:
    """(posizione media, varianza) da quote [crit, neu, fav]."""
    mean = shares[2] - shares[0]
    return mean, shares[2] + shares[0] - mean ** 2


def topic_differences(counts: np.ndarray, negatives: np.ndarray, period_idx: np.ndarray) -> np.ndarray:
    """counts: articoli × [crit, neu, fav]; negatives: commenti con emozione negativa per articolo; period_idx: periodo dell'articolo.
    Restituisce [Δ media, Δ varianza, Δ emozioni negative] osservato − atteso."""
    n_art = counts.sum(axis=1)
    n = n_art.sum()
    weights = np.bincount(period_idx, weights=n_art, minlength=len(PERIODS)) / n
    obs_mean, obs_var = shares_metrics(counts.sum(axis=0) / n)
    exp_mean, exp_var = shares_metrics(weights @ period_shares)
    return np.array([obs_mean - exp_mean, obs_var - exp_var, negatives.sum() / n - weights @ period_negative])


rng = np.random.default_rng(SEED)
period_pos = {p: i for i, p in enumerate(PERIODS)}
topic_rows = []
for topic, g in by_topic.groupby("topic_name"):
    art = (g.groupby(["article_id", "stance"], observed=False).size().unstack(fill_value=0)[STANCES]
           .join(g.groupby("article_id").agg(negatives=("negative", "sum"), period=("period", "first"))))
    art = art[art[STANCES].sum(axis=1) > 0]
    counts, negatives = art[STANCES].to_numpy(), art["negatives"].to_numpy()
    period_idx = art["period"].astype(str).map(period_pos).to_numpy()
    point = topic_differences(counts, negatives, period_idx)
    boot = np.array([topic_differences(counts[i], negatives[i], period_idx[i])
                     for i in (rng.integers(0, len(art), len(art)) for _ in range(N_BOOT))])
    lo, hi = np.quantile(boot, [0.025, 0.975], axis=0)
    row = {"topic": topic, "articles": len(art), "comments": int(counts.sum()),
           "share_2025": g["period"].astype(str).eq("2025-presente").mean(),
           **{s: counts[:, k].sum() / counts.sum() for k, s in enumerate(STANCES)}}
    for k, metric in enumerate(["d_mean", "d_variance", "d_negative"]):
        row |= {metric: point[k], f"{metric}_low": lo[k], f"{metric}_high": hi[k]}
    topic_rows.append(row)
topic_pol = pd.DataFrame(topic_rows).sort_values("d_mean").reset_index(drop=True)
topic_pol.to_csv(OUT_TOPIC_POLARIZATION, index=False)
log.info("Scritto %s", OUT_TOPIC_POLARIZATION.relative_to(BASE_DIR))

panels = [("d_mean", "Δ mean stance\n(> 0 = more favorable)"), ("d_variance", "Δ stance variance\n(> 0 = more polarized)"),
          ("d_negative", "Δ share of negative emotions\n(> 0 = more negative)")]
fig, axes = plt.subplots(1, 3, figsize=(14, 0.38 * len(topic_pol) + 1.5), sharey=True)
y = np.arange(len(topic_pol))
for ax, (metric, title) in zip(axes, panels):
    significant = (topic_pol[f"{metric}_low"] > 0) | (topic_pol[f"{metric}_high"] < 0)
    ax.errorbar(topic_pol[metric], y, xerr=[topic_pol[metric] - topic_pol[f"{metric}_low"], topic_pol[f"{metric}_high"] - topic_pol[metric]],
                fmt="none", ecolor="#b8b8b0", elinewidth=1.2)
    ax.scatter(topic_pol[metric], y, color=np.where(significant, "#1f1f1f", "#b8b8b0"), zorder=3, s=22)
    ax.axvline(0, color="#777", lw=1)
    ax.set_title(title, fontsize=9)
axes[0].set_yticks(y, [f"{t} ({a})" for t, a in zip(topic_pol["topic"], topic_pol["articles"])], fontsize=8)
axes[0].invert_yaxis()
fig.suptitle("Reader comments by article topic, compared with the average of the same period (95% CI, article bootstrap)",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "polarization_by_topic.png", dpi=150, bbox_inches="tight")
plt.show()

Una riga per topic (tra parentesi gli articoli), ordinati dall'alto dal più critico al più favorevole rispetto al proprio periodo; tre pannelli per le tre misure. Ogni punto è la differenza tra il topic e la media dei periodi in cui è stato pubblicato, con l'intervallo di confidenza; punto nero = intervallo che non tocca lo 0 (differenza non dovuta a pochi articoli), punto grigio = differenza incerta. Le quote grezze (crit, neu, fav) e la quota di commenti del 2025+ di ogni topic sono in comments_polarization_by_topic.csv.

In [ ]:
# Emozioni per topic: quote grezze sui commenti pertinenti; dominante = emozione più frequente escludendo "none"
topic_emotions = pd.crosstab(by_topic["topic_name"], by_topic["llm_emotion"], normalize="index").reindex(columns=EMOTIONS, fill_value=0)
EMOTIONS_ONLY = [e for e in EMOTIONS if e != "none"]
dominant = topic_emotions[EMOTIONS_ONLY].idxmax(axis=1)
topic_emotions = topic_emotions.assign(_dom=dominant.map(EMOTIONS_ONLY.index),
                                       _share=topic_emotions[EMOTIONS_ONLY].max(axis=1)).sort_values(["_dom", "_share"], ascending=[True, False])
dominant = dominant[topic_emotions.index]
topic_emotions = topic_emotions[EMOTIONS]

fig, ax = plt.subplots(figsize=(8, 0.38 * len(topic_emotions) + 1.5))
im = ax.imshow(topic_emotions.to_numpy(), cmap="Reds", vmin=0, vmax=0.5, aspect="auto")
for i, topic in enumerate(topic_emotions.index):
    for j, emo in enumerate(EMOTIONS):
        v = topic_emotions.iat[i, j]
        ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=7.5, color="white" if v > 0.3 else "#1f1f1f",
                fontweight="bold" if emo == dominant[topic] else "normal")
    j = EMOTIONS.index(dominant[topic])
    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="#1f1f1f", linewidth=1.5))
ax.set_xticks(range(len(EMOTIONS)), EMOTIONS, fontsize=8)
ax.set_yticks(range(len(topic_emotions)), topic_emotions.index, fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.6, label="Share of the topic's comments", format=plt.matplotlib.ticker.PercentFormatter(1))
ax.set_title("Emotions by article topic (box = dominant emotion, excluding none)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "polarization_emotion_by_topic.png", dpi=150, bbox_inches="tight")
plt.show()

Una riga per topic, una colonna per emozione. Il riquadro segna l'emozione dominante del topic, escludendo none (commenti senza emozione netta), che altrimenti prevarrebbe in metà dei topic. Righe ordinate per emozione dominante e poi per la sua quota. Sono quote grezze, non corrette per il periodo: i topic del 2025+ ereditano le emozioni di quel periodo.